# Phase 0 — Setup & Configuration

This phase initializes the environment and defines the source and pipeline configuration for the
LPI Connecting to Compete 2025 ingestion pipeline.

The purpose of this phase is to prepare the notebook for all downstream processing without
performing any PDF extraction, page classification, structural parsing, duplicate detection,
or chunking yet.

The phase establishes:

- Required Python libraries and runtime environment
- Source PDF location
- Project and output directories
- Source/document metadata
- Pipeline configuration
- Source file fingerprinting for reproducibility

No transformation of the PDF content is performed in this phase.

In [1]:
# =========================================================
# 0.1 Imports & Environment
# =========================================================

from pathlib import Path
import re
import json
import hashlib
import unicodedata
import difflib

import pandas as pd
import numpy as np
import pymupdf

print("Imports loaded successfully.")
print("pandas :", pd.__version__)
print("numpy  :", np.__version__)

Imports loaded successfully.
pandas : 3.0.5
numpy  : 2.2.4


In [2]:
# =========================================================
# 0.2 Locate the source PDF
# =========================================================

PDF_RELATIVE_PATH = Path("Data/raw/unstructured/LPI.pdf")

search_roots = [Path.cwd(), *Path.cwd().parents]

candidates = [
    root / PDF_RELATIVE_PATH
    for root in search_roots
]

PDF_PATH = next(
    (path for path in candidates if path.exists()),
    None,
)

if PDF_PATH is None:
    raise FileNotFoundError(
        "LPI Connecting to Compete 2025 PDF was not found.\n\n"
        "Looked in:\n"
        + "\n".join(f"  {path}" for path in candidates)
    )

print("PDF found successfully.")
print("PDF path :", PDF_PATH)

PDF found successfully.
PDF path : c:\Users\terbo\Desktop\Project\Data\raw\unstructured\LPI.pdf


In [3]:
# =========================================================
# 1.3 Open the source PDF
# =========================================================

doc = pymupdf.open(PDF_PATH)

print("Document opened successfully.")
print("Document name :", PDF_PATH.name)
print("Total pages   :", len(doc))

Document opened successfully.
Document name : LPI.pdf
Total pages   : 30


In [4]:
# =========================================================
# 1.4 Inspect PDF metadata
# =========================================================

pdf_metadata = doc.metadata

print("PDF metadata:")
for key, value in pdf_metadata.items():
    print(f"{key:12}: {value}")

PDF metadata:
format      : PDF 1.7
title       : 
author      : 
subject     : 
keywords    : 
creator     : Aspose Pty Ltd.
producer    : Aspose.PDF for Java 25.8
creationDate: D:20260422111837-04'00'
modDate     : D:20260422145205-04'00'
trapped     : 
encryption  : None


In [7]:
# =========================================================
# 1.5 Define project and output paths
# =========================================================

PROJECT_ROOT = PDF_PATH.parents[3]

PROCESSED_RAG_DIR = PROJECT_ROOT / "Data" / "processed" / "rag"

EXTRACTED_DIR = PROCESSED_RAG_DIR / "extracted"
CLEANED_DIR = PROCESSED_RAG_DIR / "cleaned"
CHUNKS_DIR = PROCESSED_RAG_DIR / "chunks"

for directory in (
    EXTRACTED_DIR,
    CLEANED_DIR,
    CHUNKS_DIR,
):
    directory.mkdir(parents=True, exist_ok=True)

print("Project root     :", PROJECT_ROOT)
print("Processed RAG    :", PROCESSED_RAG_DIR)
print("Extracted dir    :", EXTRACTED_DIR)
print("Cleaned dir      :", CLEANED_DIR)
print("Chunks dir       :", CHUNKS_DIR)

Project root     : c:\Users\terbo\Desktop\Project
Processed RAG    : c:\Users\terbo\Desktop\Project\Data\processed\rag
Extracted dir    : c:\Users\terbo\Desktop\Project\Data\processed\rag\extracted
Cleaned dir      : c:\Users\terbo\Desktop\Project\Data\processed\rag\cleaned
Chunks dir       : c:\Users\terbo\Desktop\Project\Data\processed\rag\chunks


In [9]:
# =========================================================
# 1.6 Define source and pipeline configuration
# =========================================================

SOURCE_NAME = "LPI_Connecting_to_Compete"
SOURCE_FILE = PDF_PATH.name

DOCUMENT_TYPE = "lpi_connecting_to_compete_2025"
REAL_OR_SYNTHETIC = "real"

DEPARTMENT = "Operations"  # owner department; must exist in the departments table (ADR-006)
SENSITIVITY_LEVEL = 1
VERSION_DATE = "2025"

SOURCE_DESCRIPTION = (
    "Connecting to Compete 2025: "
    "Trade Logistics in the Global Economy — "
    "The New Logistics Performance Indicators 2.0"
)

MIN_CHUNK_CHARS = 300
MAX_CHUNK_CHARS = 2000

source_metadata = {
    "source_name": SOURCE_NAME,
    "source_file": SOURCE_FILE,
    "document_type": DOCUMENT_TYPE,
    "real_or_synthetic": REAL_OR_SYNTHETIC,
    "department": DEPARTMENT,
    "sensitivity_level": SENSITIVITY_LEVEL,
    "version_date": VERSION_DATE,
}

print("Source configuration:")
for key, value in source_metadata.items():
    print(f"{key:20}: {value}")

print("\nChunk configuration:")
print("MIN_CHUNK_CHARS :", MIN_CHUNK_CHARS)
print("MAX_CHUNK_CHARS :", MAX_CHUNK_CHARS)

Source configuration:
source_name         : LPI_Connecting_to_Compete
source_file         : LPI.pdf
document_type       : lpi_connecting_to_compete_2025
real_or_synthetic   : real
department          : Trade, Competition, and Business
sensitivity_level   : 1
version_date        : 2025

Chunk configuration:
MIN_CHUNK_CHARS : 300
MAX_CHUNK_CHARS : 2000


In [10]:
def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    sha256 = hashlib.sha256()

    with path.open("rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            sha256.update(chunk)

    return sha256.hexdigest()


SOURCE_SHA256 = sha256_file(PDF_PATH)
SOURCE_FILE_SIZE = PDF_PATH.stat().st_size

print("Source file fingerprint:")
print("File :", PDF_PATH.name)
print("Size :", SOURCE_FILE_SIZE, "bytes")
print("SHA256:", SOURCE_SHA256)

Source file fingerprint:
File : LPI.pdf
Size : 1047039 bytes
SHA256: 3729994b68d9e34fdb1740c5d69260fb019c0461d86c72b8615e356847015171


In [11]:
assert PDF_PATH.exists(), "Source PDF does not exist."
assert len(doc) > 0, "PDF contains no pages."

assert SOURCE_NAME
assert SOURCE_FILE == PDF_PATH.name
assert DOCUMENT_TYPE == "lpi_connecting_to_compete_2025"
assert REAL_OR_SYNTHETIC == "real"
assert DEPARTMENT == "Operations"
assert SENSITIVITY_LEVEL in [1, 2, 3]
assert VERSION_DATE == "2025"

assert MIN_CHUNK_CHARS > 0
assert MAX_CHUNK_CHARS > MIN_CHUNK_CHARS

assert len(SOURCE_SHA256) == 64
assert SOURCE_FILE_SIZE > 0

print("✅ Phase 1 validation passed.")
print("Pages :", len(doc))
print("Source:", SOURCE_NAME)
print("SHA256:", SOURCE_SHA256)

✅ Phase 1 validation passed.
Pages : 30
Source: LPI_Connecting_to_Compete
SHA256: 3729994b68d9e34fdb1740c5d69260fb019c0461d86c72b8615e356847015171


# Phase 2 — Page Inventory

This phase creates a raw page-level inventory of the source PDF.

The purpose is to inspect the document page by page before applying any
classification, structural parsing, table/figure extraction, cleaning,
deduplication, or chunking logic.

For each physical PDF page, the inventory records:

- Page number
- Character count
- Word count
- Line count
- Raw extracted text

At this stage, the extracted text is preserved as-is. No normalization,
cleaning, paragraph reconstruction, or content filtering is performed.

The resulting page inventory will be used as the foundation for the next
phase, where each page is classified according to its actual content and
role in the document.

In [12]:
page_records = []

for page_idx in range(len(doc)):
    page = doc[page_idx]
    text = page.get_text("text") or ""

    page_records.append({
        "page_number": page_idx + 1,
        "char_count": len(text),
        "word_count": len(text.split()),
        "line_count": len(text.splitlines()),
        "text": text,
    })

pages_df = pd.DataFrame(page_records)

print("Page inventory created successfully.")
print("Rows :", len(pages_df))
print("Columns:", list(pages_df.columns))

Page inventory created successfully.
Rows : 30
Columns: ['page_number', 'char_count', 'word_count', 'line_count', 'text']


In [13]:
print("Page inventory statistics:\n")

print("Total pages :", len(pages_df))
print("Total chars :", pages_df["char_count"].sum())
print("Total words :", pages_df["word_count"].sum())
print("Total lines :", pages_df["line_count"].sum())

print("\nCharacter count:")
print(pages_df["char_count"].describe())

print("\nWord count:")
print(pages_df["word_count"].describe())

Page inventory statistics:

Total pages : 30
Total chars : 81116
Total words : 11795
Total lines : 2219

Character count:
count      30.000000
mean     2703.866667
std      1535.624319
min         0.000000
25%      1495.250000
50%      3097.000000
75%      3769.250000
max      5623.000000
Name: char_count, dtype: float64

Word count:
count     30.000000
mean     393.166667
std      226.299174
min        0.000000
25%      214.500000
50%      466.500000
75%      531.000000
max      836.000000
Name: word_count, dtype: float64


In [14]:
preview_df = pages_df.copy()

preview_df["text_preview"] = (
    preview_df["text"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
    .str[:250]
)

display(
    preview_df[
        ["page_number", "char_count", "word_count", "line_count", "text_preview"]
    ]
)

,page_number,char_count,word_count,line_count,text_preview
0,1,228,28,11,Connecting to Compete 2025 Trade Logistics in ...
1,2,220,25,5,Connecting to Compete 2025 The New Logistics P...
2,3,2085,318,28,"© 2026 The World Bank 1818 H Street NW, Washin..."
3,4,1465,222,71,CONNECTING TO COMPETE 2025 THE NEW LOGISTICS ...
4,5,712,112,26,iv CONNECTING TO COMPETE 2025 THE NEW LOGISTI...
5,6,2551,372,62,CONNECTING TO COMPETE 2025 THE NEW LOGISTICS ...
6,7,375,45,18,vi CONNECTING TO COMPETE 2025 THE NEW LOGISTI...
7,8,1586,212,36,CONNECTING TO COMPETE 2025 THE NEW LOGISTICS ...
8,9,0,0,0,
9,10,3590,511,84,CONNECTING TO COMPETE 2025 THE NEW LOGISTICS ...


In [15]:
special_pages_df = pages_df[
    (pages_df["char_count"] == 0) |
    (pages_df["char_count"] < 500) |
    (pages_df["char_count"] > 5000)
].copy()

display(
    special_pages_df[
        ["page_number", "char_count", "word_count", "line_count"]
    ]
)

,page_number,char_count,word_count,line_count
0,1,228,28,11
1,2,220,25,5
6,7,375,45,18
8,9,0,0,0
27,28,5623,836,146


In [16]:
inspection_pages = [1, 2, 3, 5, 6, 7, 8, 9, 10, 28, 29, 30]

for page_number in inspection_pages:
    row = pages_df.loc[pages_df["page_number"] == page_number].iloc[0]

    print("=" * 100)
    print(f"PAGE {page_number}")
    print(f"chars={row['char_count']} | words={row['word_count']} | lines={row['line_count']}")
    print("-" * 100)

    text_preview = row["text"].strip()

    if not text_preview:
        print("[EMPTY PAGE]")
    else:
        print(text_preview[:1200])

    print()

PAGE 1
chars=228 | words=28 | lines=11
----------------------------------------------------------------------------------------------------
Connecting
to
Compete
2025
Trade Logistics in the 
Global Economy
The New Logistics Performance Indicators 2.0 
Public Disclosure Authorized
Public Disclosure Authorized
Public Disclosure Authorized
Public Disclosure Authorized

PAGE 2
chars=220 | words=25 | lines=5
----------------------------------------------------------------------------------------------------
Connecting to Compete 2025
The New Logistics Performance Indicators 2.0
A reproducibility package is available for this 
book in the Reproducible Research Repository at 
https://reproducibility.worldbank.org/catalog/514

PAGE 3
chars=2085 | words=318 | lines=28
----------------------------------------------------------------------------------------------------
© 2026 The World Bank
1818 H Street NW, Washington DC 20433
Telephone: 202-­473-­1000; Internet: www.worldbank.org
Some rights re